In [1]:
import pandas as pd
import pickle
import importlib
import sys
import os
from syllabification import tokenize, syllabify_sentences
from markov_models import MarkovModel 
from helpers import get_word_data, get_info_rate, update_values_in_csv
import warnings
import re

# define parameters
language = "YUE"
input_type = "words"

In [6]:
n_values = [1, 2, 3, 4]  # For bigram, trigram, and 4-gram models
markov_models = {}

for n in n_values:
    
    # Create and build the Markov model
    model = MarkovModel(n)

    if input_type == "sentences": 
        # Load the paired data
        with open(f"produced_data/{language}/sentence_pairs.pkl", "rb") as f: 
            sentence_pairs = pickle.load(f)

        # Merge all transcribed (syllabified) sentences into one list
        merged_sentences = []
        for tokenized, transcribed in sentence_pairs:
            merged_sentences.extend(transcribed)

        model.build(merged_sentences, input_type)

    elif input_type == "words": 
        if language == "FRA":
            path = "Z:/data/FRA/Lexique383.tsv"  
        elif language == "JPN":
            path = "Z:/data/JPN/jpn.txt"
        elif language == "CMN":
            path = "Z:/data/CMN/cmn.txt"
        elif language == "VIE":
            path = "Z:/data/VIE/vie.txt"
        elif language == "YUE":
            path = "Z:/data/YUE/yue.txt"
        else:
            raise ValueError(f"Unsupported language: {language}")

        # Load the data
        words = get_word_data(path)
       
        print(f"\nTraining a Markov Model with n = {n}:")
        # Build the markov model
        model.build(words, input_type)


    # Compute the conditional entropy (information density)
    info_density = model.compute_conditional_entropy()
    print(f"Information Density: {info_density}")

    # Compute the information rate (bits per second)
    info_rate = get_info_rate(info_density, language)
    print(f"Information Rate:")
    print(f"  {', '.join(map(lambda x: f'{x:.4f}', info_rate[:5]))} ... (total {len(info_rate)} values)\n")
    
    # Update the CSV file with the computed info_density and info_rate
    update_values_in_csv(language, info_density, n, 'ID')
    update_values_in_csv(language, info_rate, n, 'IR')

    # Store model for later use 
    markov_models[n] = model

    # Display exactly 3 examples
    example_count = 0
    print("\nExample probabilities (p(x, y)):")

    for (prefix, suffix), p_xy in model.normalized_probs.items():
        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
        example_count += 1
        if example_count == 3:
            break
    
    # Save the model to a file
    model.save_model(language, input_type)

Language: YUE

Training a Markov Model with n = 1:
Information Density: 7.971067822204827
Information Rate:
  40.6011, 41.9232, 40.0977, 45.6040, 43.1988 ... (total 150 values)

Updated ID_unigram_esidaine
Updated IR_unigram_esidaine

Example probabilities (p(x, y)):
p(() -> nei5) = 0.0454
p(() -> ŋQ5) = 0.0423
p(() -> A1) = 0.0379

✅ Saved 1-gram model to 'produced_data/YUE/'
Language: YUE

Training a Markov Model with n = 2:
Information Density: 1.894666596178111
Information Rate:
  9.6506, 9.9648, 9.5309, 10.8398, 10.2681 ... (total 150 values)

Updated ID_bigram_esidaine
Updated IR_bigram_esidaine

Example probabilities (p(x, y)):
p(('ŋQ5',) -> tei6) = 0.9974
p(('jI4',) -> kA1) = 0.8623
p(('Sɐn1',) -> hɐi6) = 0.9259

✅ Saved 2-gram model to 'produced_data/YUE/'
Language: YUE

Training a Markov Model with n = 3:
Information Density: 0.06232048549426686
Information Rate:
  0.3174, 0.3278, 0.3135, 0.3565, 0.3377 ... (total 150 values)

Updated ID_trigram_esidaine
Updated IR_trigram_es

In [5]:
# prepare transcribed sentences 

if language == "FRA" and input_type == "sentences": 
    path = "Z:/data/FRA/french_sentences.txt"  # Use the mounted drive letter

    # Read each line as a sentence
    tokenized_sentences = []
    transcribed_sentences = []


    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            sentence = line.strip()
            if sentence:
                # Tokenize
                tokenized_sentence = tokenize(sentence)
                print(tokenized_sentence)
                tokenized_sentences.append(tokenized_sentence)
                
                # Syllabify
                transcribed_sentence = syllabify_sentences(tokenized_sentence, language="FRA")
                print(transcribed_sentence)
                if transcribed_sentence: 
                    transcribed_sentences.append(transcribed_sentence)

            # Optional: limit for testing
            if i >= 20:
                break


    # show the entries 
    for i, sentence in enumerate(transcribed_sentences[:20]):
        print(f"Sentence {i+1}: {sentence}")

else: 
    warnings.warn("Warning: The specified language is not available yet")


# Save to .pkl
paired_sentences = list(zip(tokenized_sentences, transcribed_sentences))

with open("produced_data/{language}/preprocessed_{input_type}.pkl", "wb") as f:
    pickle.dump(paired_sentences, f)

print(f"✅ Saved tokenized and transcribed sentences to 'produced_data/{language}'")


C:\Users\emill\AppData\Local\Temp\ipykernel_34572\1726129514.py:36: UserWarning: Warning: The specified language is not available yet
  warnings.warn("Warning: The specified language is not available yet")


NameError: name 'tokenized_sentences' is not defined